In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read gold layer tables
fact_invoices = spark.table("automobile_catalog.003_gold.fact_invoices")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")

# Define MTD periods (using latest available data: January 2026 vs December 2025)
current_mtd_start = '2026-01-01'
current_mtd_end = '2026-01-30'
previous_mtd_start = '2025-12-01'
previous_mtd_end = '2025-12-30'

# Filter completed orders and join with dim_store
completed_orders = (
    fact_invoices
    .filter(F.col("order_status") == "COMPLETED")
    .join(dim_store, ["store_id", "manager_id"], "inner")
)

# Calculate current MTD metrics
current_mtd = completed_orders.filter(
    (F.col("invoice_date") >= current_mtd_start) & 
    (F.col("invoice_date") <= current_mtd_end)
).groupBy(
    "store_id", "store_name", "manager_id", "manager_name"
).agg(
    F.sum("invoice_amount").alias("current_mtd_revenue"),
    F.count("order_id").alias("current_mtd_orders")
)

# Calculate previous MTD metrics
previous_mtd = completed_orders.filter(
    (F.col("invoice_date") >= previous_mtd_start) & 
    (F.col("invoice_date") <= previous_mtd_end)
).groupBy(
    "store_id", "store_name", "manager_id", "manager_name"
).agg(
    F.sum("invoice_amount").alias("previous_mtd_revenue"),
    F.count("order_id").alias("previous_mtd_orders")
)

# Combine current and previous MTD
mtd_comparison = current_mtd.join(
    previous_mtd,
    ["store_id", "store_name", "manager_id", "manager_name"],
    "full_outer"
).fillna(0, subset=["current_mtd_revenue", "current_mtd_orders", "previous_mtd_revenue", "previous_mtd_orders"])

# Calculate performance metrics
mtd_kpi = mtd_comparison.withColumn(
    "revenue_change",
    F.col("current_mtd_revenue") - F.col("previous_mtd_revenue")
).withColumn(
    "revenue_change_pct",
    F.when(F.col("previous_mtd_revenue") > 0,
           ((F.col("current_mtd_revenue") - F.col("previous_mtd_revenue")) / F.col("previous_mtd_revenue") * 100)
    ).otherwise(None)
).withColumn(
    "orders_change",
    F.col("current_mtd_orders") - F.col("previous_mtd_orders")
).withColumn(
    "orders_change_pct",
    F.when(F.col("previous_mtd_orders") > 0,
           ((F.col("current_mtd_orders") - F.col("previous_mtd_orders")) / F.col("previous_mtd_orders") * 100)
    ).otherwise(None)
).select(
    "store_id",
    "store_name",
    "manager_id",
    "manager_name",
    F.round("current_mtd_revenue", 2).alias("current_mtd_revenue"),
    F.col("current_mtd_orders"),
    F.round("previous_mtd_revenue", 2).alias("previous_mtd_revenue"),
    F.col("previous_mtd_orders"),
    F.round("revenue_change", 2).alias("revenue_change"),
    F.round("revenue_change_pct", 2).alias("revenue_change_pct"),
    F.col("orders_change"),
    F.round("orders_change_pct", 2).alias("orders_change_pct")
).orderBy(F.desc("current_mtd_revenue"))

# Display results
display(mtd_kpi)

In [0]:
from pyspark.sql import functions as F

# Read gold layer tables
fact_orders = spark.table("automobile_catalog.003_gold.fact_orders")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")

# Join fact_orders with dim_store
orders_with_store = fact_orders.join(dim_store, "store_id", "inner")

# Calculate average days in shop by store and service type
avg_days_in_shop = (
    orders_with_store
    .filter(F.col("days_in_shop").isNotNull())
    .groupBy("store_id", "store_name", "service_type")
    .agg(
        F.round(F.avg("days_in_shop"), 2).alias("avg_days_in_shop"),
        F.count("order_id").alias("total_orders")
    )
    .orderBy("store_id", "service_type")
)

display(avg_days_in_shop)

In [0]:
from pyspark.sql import functions as F

# Read gold layer tables
fact_survey_responses = spark.table("automobile_catalog.003_gold.fact_survey_responses")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")

# Join fact_survey_responses with dim_store
survey_with_store = fact_survey_responses.join(dim_store, "store_id", "inner")

# Calculate survey coverage by store
survey_coverage = (
    survey_with_store
    .groupBy("store_id", "store_name")
    .agg(
        F.count("survey_id").alias("surveys_sent"),
        F.sum(F.when(F.col("responded_flag") == True, 1).otherwise(0)).alias("surveys_responded"),
        F.round(
            (F.sum(F.when(F.col("responded_flag") == True, 1).otherwise(0)) / F.count("survey_id")) * 100,
            2
        ).alias("response_rate_pct")
    )
    .orderBy(F.desc("response_rate_pct"))
)

display(survey_coverage)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read gold layer tables
fact_survey_responses = spark.table("automobile_catalog.003_gold.fact_survey_responses")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")

# Join fact_survey_responses with dim_store, filter for responses only
survey_with_store = (
    fact_survey_responses
    .filter(F.col("responded_flag") == True)
    .join(dim_store, "store_id", "inner")
)

# Calculate average scores by store
survey_scores = (
    survey_with_store
    .groupBy("store_id", "store_name")
    .agg(
        F.round(F.avg("delivered_on_time_rating"), 2).alias("avg_delivered_on_time"),
        F.round(F.avg("work_quality_rating"), 2).alias("avg_work_quality"),
        F.round(F.avg("cleanliness_rating"), 2).alias("avg_cleanliness"),
        F.round(F.avg("communication_rating"), 2).alias("avg_communication"),
        F.round(F.avg("overall_satisfaction_rating"), 2).alias("avg_overall_satisfaction"),
        F.count("survey_id").alias("total_responses")
    )
)

# Add ranking by overall satisfaction
window_spec = Window.orderBy(F.desc("avg_overall_satisfaction"))
survey_scores_ranked = survey_scores.withColumn(
    "satisfaction_rank",
    F.rank().over(window_spec)
).orderBy("satisfaction_rank")

display(survey_scores_ranked)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read gold layer tables
fact_invoices = spark.table("automobile_catalog.003_gold.fact_invoices")
fact_budget = spark.table("automobile_catalog.003_gold.fact_budget")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")

# Calculate monthly revenue from fact_invoices (completed orders only)
monthly_revenue = (
    fact_invoices
    .filter(F.col("order_status") == "COMPLETED")
    .join(
        dim_store.select(
            "store_id",
            F.col("store_name").alias("store_name_dim"),
            F.col("manager_id").alias("manager_id_dim"),
            F.col("manager_name").alias("manager_name_dim")
        ),
        "store_id",
        "inner"
    )
    .groupBy("invoice_month", "manager_id", "manager_name_dim", "store_id", "store_name_dim")
    .agg(F.sum("invoice_amount").alias("actual_revenue"))
)

# Combine revenue and budget from fact_budget
revenue_vs_budget = (
    monthly_revenue
    .join(
        fact_budget.select(F.col("store_id").alias("budget_store_id"), "budget_month", "budget_amount"),
        (monthly_revenue.store_id == F.col("budget_store_id")) &
        (monthly_revenue.invoice_month == fact_budget.budget_month),
        "left"
    )
    .withColumn(
        "variance",
        F.col("actual_revenue") - F.coalesce(F.col("budget_amount"), F.lit(0))
    )
    .withColumn(
        "achievement_pct",
        F.when(
            F.col("budget_amount") > 0,
            F.round((F.col("actual_revenue") / F.col("budget_amount")) * 100, 2)
        ).otherwise(None)
    )
    .select(
        "invoice_month",
        "manager_id",
        F.col("manager_name_dim").alias("manager_name"),
        "store_id",
        F.col("store_name_dim").alias("store_name"),
        F.round("actual_revenue", 2).alias("actual_revenue"),
        F.round("budget_amount", 2).alias("budget_amount"),
        F.round("variance", 2).alias("variance"),
        "achievement_pct"
    )
)

# Rank managers by budget achievement
window_spec = Window.partitionBy("invoice_month").orderBy(F.desc("achievement_pct"))
revenue_vs_budget_ranked = revenue_vs_budget.withColumn(
    "achievement_rank",
    F.rank().over(window_spec)
).orderBy("invoice_month", "achievement_rank")

display(revenue_vs_budget_ranked)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read gold layer tables
fact_orders = spark.table("automobile_catalog.003_gold.fact_orders")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")
dim_technician = spark.table("automobile_catalog.003_gold.dim_technician")

# Filter completed orders with promised and actual completion dates
completed_orders = (
    fact_orders
    .filter(
        (F.col("order_status") == "COMPLETED") &
        F.col("promised_delivery_datetime").isNotNull() &
        F.col("actual_delivery_datetime").isNotNull()
    )
    .join(dim_store.select("store_id", F.col("store_name").alias("store_name_dim")), "store_id", "inner")
    .join(dim_technician, ["technician_id", "store_id"], "inner")
)

# Calculate completion time accuracy for each technician
technician_accuracy = (
    completed_orders
    .withColumn(
        "delivery_diff_days",
        F.abs(F.datediff(F.col("actual_delivery_datetime"), F.col("promised_delivery_datetime")))
    )
    .withColumn(
        "is_on_time",
        F.when(F.col("actual_delivery_datetime") <= F.col("promised_delivery_datetime"), 1).otherwise(0)
    )
    .groupBy("technician_id", "technician_name", "store_id", "store_name")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("is_on_time").alias("on_time_deliveries"),
        F.round(
            (F.sum("is_on_time") / F.count("order_id")) * 100,
            2
        ).alias("on_time_percentage"),
        F.round(F.avg("delivery_diff_days"), 2).alias("avg_delivery_variance_days")
    )
    .filter(F.col("total_orders") >= 10)  # Only technicians with at least 10 orders
)

# Rank technicians by on-time percentage
window_spec = Window.orderBy(F.desc("on_time_percentage"), F.asc("avg_delivery_variance_days"))
technician_ranked = technician_accuracy.withColumn(
    "accuracy_rank",
    F.rank().over(window_spec)
).orderBy("accuracy_rank")

# Display top 20 technicians
display(technician_ranked.limit(20))


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read gold layer tables
fact_invoices = spark.table("automobile_catalog.003_gold.fact_invoices")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")

# Filter completed orders
revenue_data = (
    fact_invoices
    .filter(F.col("order_status") == "COMPLETED")
    .join(dim_store, "store_id", "inner")
)

# Get latest year in the dataset
latest_year_row = revenue_data.agg(F.max("invoice_year")).first()
latest_year = latest_year_row[0] if latest_year_row else None

if latest_year:
    current_year = latest_year
    previous_year = latest_year - 1
    
    # Calculate current YTD revenue
    current_ytd = (
        revenue_data
        .filter(F.col("invoice_year") == current_year)
        .groupBy("store_id", "store_name")
        .agg(F.sum("invoice_amount").alias("current_ytd_revenue"))
    )
    
    # Calculate previous YTD revenue
    previous_ytd = (
        revenue_data
        .filter(F.col("invoice_year") == previous_year)
        .groupBy("store_id", "store_name")
        .agg(F.sum("invoice_amount").alias("previous_ytd_revenue"))
    )
    
    # Combine and calculate growth
    ytd_growth = (
        current_ytd
        .join(previous_ytd, ["store_id", "store_name"], "full_outer")
        .fillna(0, subset=["current_ytd_revenue", "previous_ytd_revenue"])
        .withColumn(
            "ytd_growth",
            F.col("current_ytd_revenue") - F.col("previous_ytd_revenue")
        )
        .withColumn(
            "ytd_growth_pct",
            F.when(
                F.col("previous_ytd_revenue") > 0,
                F.round((F.col("ytd_growth") / F.col("previous_ytd_revenue")) * 100, 2)
            ).otherwise(None)
        )
        .select(
            "store_id",
            "store_name",
            F.round("current_ytd_revenue", 2).alias("current_ytd_revenue"),
            F.round("previous_ytd_revenue", 2).alias("previous_ytd_revenue"),
            F.round("ytd_growth", 2).alias("ytd_growth"),
            "ytd_growth_pct"
        )
    )
    
    # Rank stores by YTD growth percentage
    window_spec = Window.orderBy(F.desc("ytd_growth_pct"))
    ytd_growth_ranked = ytd_growth.withColumn(
        "growth_rank",
        F.rank().over(window_spec)
    ).orderBy("growth_rank")
    
    display(ytd_growth_ranked)
else:
    print("No revenue data available to calculate YTD growth")

In [0]:

from pyspark.sql import functions as F

# Read gold layer tables
fact_orders = spark.table("automobile_catalog.003_gold.fact_orders")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")

# Join fact_orders with dim_store
orders_with_store = fact_orders.join(dim_store, "store_id", "inner")

# Calculate stage-wise cycle times using pre-calculated metrics in fact_orders
stage_cycle_times = (
    orders_with_store
    .filter(
        F.col("days_to_work_start").isNotNull() &
        F.col("work_duration_days").isNotNull() &
        F.col("days_in_shop").isNotNull()
    )
    .groupBy("store_id", "store_name", "service_type")
    .agg(
        F.round(F.avg("days_to_work_start"), 2).alias("avg_intake_to_work_days"),
        F.round(F.avg("work_duration_days"), 2).alias("avg_work_to_completion_days"),
        F.round(F.avg("days_in_shop"), 2).alias("avg_total_cycle_days"),
        F.count("order_id").alias("total_orders")
    )
    .orderBy("store_id", "service_type")
)

display(stage_cycle_times)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read gold layer tables
fact_estimates = spark.table("automobile_catalog.003_gold.fact_estimates")
dim_estimator = spark.table("automobile_catalog.003_gold.dim_estimator")

# Get initial estimates with accuracy metrics
estimate_accuracy = (
    fact_estimates
    .filter(
        (F.col("is_initial_estimate") == True) &
        F.col("actual_amount").isNotNull() &
        (F.col("actual_amount") > 0)
    )
    .join(dim_estimator, "estimator_id", "inner")
    .withColumn(
        "accuracy_score",
        F.when(F.col("variance_pct") <= 5, 100)
        .when(F.col("variance_pct") <= 10, 90)
        .when(F.col("variance_pct") <= 15, 80)
        .when(F.col("variance_pct") <= 20, 70)
        .otherwise(F.greatest(F.lit(0), 100 - F.col("variance_pct")))
    )
)

# Aggregate by estimator
estimator_performance = (
    estimate_accuracy
    .groupBy("estimator_id", "estimator_name")
    .agg(
        F.count("estimate_id").alias("total_estimates"),
        F.round(F.avg("estimate_amount"), 2).alias("avg_initial_estimate"),
        F.round(F.avg("actual_amount"), 2).alias("avg_actual_amount"),
        F.round(F.avg("estimate_variance"), 2).alias("avg_variance"),
        F.round(F.avg("variance_pct"), 2).alias("avg_variance_pct"),
        F.round(F.avg("accuracy_score"), 2).alias("avg_accuracy_score"),
        F.sum(F.when(F.col("variance_pct") <= 10, 1).otherwise(0)).alias("estimates_within_10pct")
    )
    .withColumn(
        "accuracy_rate_pct",
        F.round((F.col("estimates_within_10pct") / F.col("total_estimates")) * 100, 2)
    )
)

# Rank estimators by accuracy score
window_spec = Window.orderBy(F.desc("avg_accuracy_score"))
estimator_ranked = estimator_performance.withColumn(
    "accuracy_rank",
    F.rank().over(window_spec)
).orderBy("accuracy_rank")

display(estimator_ranked)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read gold layer tables
fact_orders = spark.table("automobile_catalog.003_gold.fact_orders")
dim_store = spark.table("automobile_catalog.003_gold.dim_store")
dim_technician = spark.table("automobile_catalog.003_gold.dim_technician")

# Join fact_orders with dimensions (avoid ambiguous column references)
orders_with_metrics = (
    fact_orders
    .join(dim_store.select("store_id", F.col("store_name").alias("store_name_dim")), "store_id", "inner")
    .join(dim_technician, ["technician_id", "store_id"], "inner")
    .filter(F.col("days_in_shop").isNotNull())
    .withColumn(
        "work_month",
        F.date_format(F.col("vehicle_in_datetime"), "yyyy-MM")
    )
)

# Calculate technician workload per month
technician_workload = (
    orders_with_metrics
    .groupBy("work_month", "store_id", "store_name", "technician_id", "technician_name")
    .agg(
        F.count("order_id").alias("orders_handled"),
        F.round(F.sum("days_in_shop"), 2).alias("total_days_in_shop"),
        F.round(F.avg("days_in_shop"), 2).alias("avg_days_per_order")
    )
)

# Rank technicians by workload within each month
window_spec = Window.partitionBy("work_month").orderBy(F.desc("orders_handled"), F.desc("total_days_in_shop"))
technician_workload_ranked = technician_workload.withColumn(
    "workload_rank",
    F.rank().over(window_spec)
).orderBy("work_month", "workload_rank")

# Display top 20 technicians per month
display(technician_workload_ranked.filter(F.col("workload_rank") <= 20))